# Robust Curve Regression with Outliers

Companion notebook for the Towards Data Science article **“Don’t Let Messy Data Bend Your Curves.”**

This notebook extends the same robust-estimator implementations used in the earlier linear-regression study to curved regression models. The estimator logic is kept fixed while the model representation changes from a straight-line basis to quadratic, cubic, and B-spline bases.

**Methods:** OLS, Huber, RANSAC, GNC-GM, GNC-TLS, and ASOR.

**Model classes:** quadratic polynomial, cubic polynomial, and quadratic B-spline.

**Contamination mechanisms:** zero-mean Gaussian replacement, biased Gaussian replacement, biased uniform replacement, and a coherent competing curve.

The default `RUN_PROFILE = "quick"` is intended for a fast smoke test. Set `RUN_PROFILE = "full"` to reproduce the 30-Monte-Carlo experiments reported with the article.

**Article:** [Don’t Let Messy Data Bend Your Curves](https://towardsdatascience.com/dont-let-messy-data-bend-your-curves/)

**Previous article in the series:** [How to Make Linear Regression Survive Outliers](https://towardsdatascience.com/how-to-make-linear-regression-survive-outliers/)

**Previous code repository:** [robust-linear-regression-outliers](https://github.com/chughtaiah/robust-linear-regression-outliers)


In [404]:
# If required:
# %pip install numpy pandas scipy

import math
import time

import numpy as np
import pandas as pd

from scipy.interpolate import BSpline
from scipy.special import expit, gamma
from scipy.stats import chi2

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


## 1. Benchmark Settings

The full benchmark uses:

- 30 Monte Carlo realizations per exact condition
- $N=100$ for the contamination sweep
- $\sigma_{\mathrm{in}}^2=1000$
- contamination ratios of $0\%$, $10\%$, $30\%$, $50\%$, $70\%$, and $90\%$
- outlier-to-inlier noise-variance ratio $\sigma_{\mathrm{out}}^2/\sigma_{\mathrm{in}}^2=10$
- biased-outlier mean shift $3\sigma_{\mathrm{in}}$
- sample sizes $N\in\{50,100,200,500,1000,2000,5000\}$
- noise variances $\sigma_{\mathrm{in}}^2\in\{10,100,1000,10000\}$

The notebook displays only the compact tables needed to support the article's quantitative claims. It does **not** generate publication figures or write CSV files.

Use `"quick"` first to verify the pipeline. Change to `"full"` to reproduce the article numbers.


In [405]:
RUN_PROFILE = "full"   # "quick" or "full"

if RUN_PROFILE not in {"quick", "full"}:
    raise ValueError("RUN_PROFILE must be either 'quick' or 'full'")

MASTER_SEED = 20261015

MODEL_NAMES = [
    "quadratic",
    "cubic",
    "bspline",
]

METHOD_NAMES = [
    "OLS",
    "Huber",
    "RANSAC",
    "GNC-GM",
    "GNC-TLS",
    "ASOR",
]

CONTAMINATION_TYPES = [
    "zero_mean_gaussian",
    "biased_gaussian",
    "biased_uniform",
    "competing_curve",
]

X_MIN = -100.0
X_MAX = 100.0

BASELINE_N = 100
BASELINE_VARIANCE = 1000.0

VARIANCE_RATIO = 10.0
BIASED_MEAN_SHIFT_SIGMA = 3.0

CONTAMINATION_RATIOS_FULL = np.array([
    0.0, 0.10, 0.30, 0.50, 0.70, 0.90
])

SAMPLE_SIZES_FULL = [
    50, 100, 200, 500, 1000, 2000, 5000
]

NOISE_VARIANCES_FULL = [
    10.0, 100.0, 1000.0, 10000.0
]

if RUN_PROFILE == "full":
    MC_RUNS = 30
    CONTAMINATION_RATIOS = CONTAMINATION_RATIOS_FULL
    SAMPLE_SIZES = SAMPLE_SIZES_FULL
    NOISE_VARIANCES = NOISE_VARIANCES_FULL
else:
    MC_RUNS = 3
    CONTAMINATION_RATIOS = np.array([0.0, 0.30, 0.50])
    SAMPLE_SIZES = [50, 100, 500]
    NOISE_VARIANCES = [100.0, 1000.0]

print({
    "profile": RUN_PROFILE,
    "MC runs": MC_RUNS,
    "contamination ratios": CONTAMINATION_RATIOS.tolist(),
})


{'profile': 'full', 'MC runs': 30, 'contamination ratios': [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]}


## 2. Model Bases

Polynomial predictors are scaled as

$$
z=\frac{x}{100}
$$

so that $x\in[-100,100]$ maps to $z\in[-1,1]$.

The spline model uses a quadratic B-spline with five interior knots, giving eight basis functions.


In [406]:
def scaled_predictor(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return x / 100.0


def polynomial_design(x, degree):
    z = scaled_predictor(x)
    return np.vander(
        z,
        N=degree + 1,
        increasing=True,
    )


SPLINE_DEGREE = 2
SPLINE_INTERIOR_KNOTS = np.linspace(
    X_MIN,
    X_MAX,
    7,
)[1:-1]

SPLINE_KNOT_VECTOR = np.concatenate([
    np.repeat(X_MIN, SPLINE_DEGREE + 1),
    SPLINE_INTERIOR_KNOTS,
    np.repeat(X_MAX, SPLINE_DEGREE + 1),
])

SPLINE_BASIS_COUNT = (
    len(SPLINE_KNOT_VECTOR)
    - SPLINE_DEGREE
    - 1
)


def bspline_design(x):
    x = np.asarray(x, dtype=float).reshape(-1)

    Phi = np.empty(
        (len(x), SPLINE_BASIS_COUNT),
        dtype=float,
    )

    for j in range(SPLINE_BASIS_COUNT):
        coefficient_vector = np.zeros(
            SPLINE_BASIS_COUNT
        )
        coefficient_vector[j] = 1.0

        basis_function = BSpline(
            SPLINE_KNOT_VECTOR,
            coefficient_vector,
            SPLINE_DEGREE,
            extrapolate=False,
        )

        Phi[:, j] = basis_function(x)

    return Phi


def design_matrix(x, model_name):
    if model_name == "quadratic":
        return polynomial_design(x, degree=2)

    if model_name == "cubic":
        return polynomial_design(x, degree=3)

    if model_name == "bspline":
        return bspline_design(x)

    raise ValueError(
        f"Unknown model: {model_name}"
    )


# Nominal coefficients chosen so the three synthetic curves have
# comparable overall signal magnitudes on [-100, 100].
TRUE_BETA = {
    "quadratic": np.array([
        20.0,
        260.0,
        120.0,
    ]),
    "cubic": np.array([
        20.0,
        220.0,
        -80.0,
        140.0,
    ]),
    "bspline": np.array([
        -220.0,
        -150.0,
        -40.0,
        90.0,
        40.0,
        180.0,
        230.0,
        290.0,
    ]),
}


def true_curve(x, model_name):
    return (
        design_matrix(x, model_name)
        @ TRUE_BETA[model_name]
    )


def number_of_coefficients(model_name):
    return design_matrix(
        np.linspace(X_MIN, X_MAX, 20),
        model_name,
    ).shape[1]


pd.DataFrame({
    "Model": MODEL_NAMES,
    "Number of coefficients": [
        number_of_coefficients(name)
        for name in MODEL_NAMES
    ],
    "RANSAC minimal sample": [
        number_of_coefficients(name)
        for name in MODEL_NAMES
    ],
})


,Model,Number of coefficients,RANSAC minimal sample
0,quadratic,3,3
1,cubic,4,4
2,bspline,8,8


## 3. Shared Least-Squares Core

OLS and the weighting-based robust estimators use the same weighted least-squares solver. The solver receives the already-constructed design matrix $\Phi$, so changing the model representation does not require changing the linear-algebra core.


In [407]:
def predict(beta, Phi):
    Phi = np.asarray(Phi, dtype=float)
    return Phi @ beta


def weighted_least_squares(
    Phi,
    y,
    weights=None,
):
    Phi = np.asarray(Phi, dtype=float)
    y = np.asarray(y, dtype=float)

    if weights is None:
        weights = np.ones(len(y))

    weights = np.asarray(
        weights,
        dtype=float,
    )

    if np.any(weights < 0):
        raise ValueError(
            "Weights must be nonnegative"
        )

    sqrt_w = np.sqrt(weights)
    Phiw = Phi * sqrt_w[:, None]
    yw = y * sqrt_w

    beta, _, rank, _ = np.linalg.lstsq(
        Phiw,
        yw,
        rcond=None,
    )

    if rank < Phi.shape[1]:
        raise np.linalg.LinAlgError(
            "Rank-deficient weighted design matrix"
        )

    return beta


def normalized_squared_residuals(
    beta,
    Phi,
    y,
    sigma,
):
    residuals = (
        np.asarray(y, dtype=float)
        - predict(beta, Phi)
    )

    return residuals**2 / sigma**2


## 4. Estimator Algorithms

The estimator settings are kept fixed across the three model classes.

### OLS


In [408]:
def fit_ols(Phi, y):
    return weighted_least_squares(
        Phi,
        y,
    )


### Huber

Benchmark settings:

- $\delta=1.35$
- tolerance $10^{-5}$
- maximum 100 IRLS iterations


In [409]:
def fit_huber(
    Phi,
    y,
    sigma,
    delta=1.35,
    tol=1e-5,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    for _ in range(max_iter):
        residuals = (
            np.asarray(y)
            - predict(beta, Phi)
        ) / sigma

        abs_residuals = np.abs(
            residuals
        )

        weights = np.ones_like(
            abs_residuals
        )

        large = abs_residuals > delta

        weights[large] = (
            delta
            / abs_residuals[large]
        )

        beta_new = weighted_least_squares(
            Phi,
            y,
            weights,
        )

        relative_change = (
            np.linalg.norm(
                beta_new - beta
            )
            / max(
                np.linalg.norm(beta),
                1e-12,
            )
        )

        beta = beta_new

        if relative_change <= tol:
            break

    return beta


### RANSAC

Benchmark settings:

- normalized-residual threshold $=3$
- confidence $=0.999$
- maximum 10,000 hypotheses
- deterministic per-fit seed

The minimal sample size $s$ equals the number of fitted coefficients. Therefore:

- quadratic polynomial: $s=3$
- cubic polynomial: $s=4$
- quadratic B-spline: $s=8$

The implementation also records the number of hypotheses evaluated, the final consensus fraction, and whether the maximum-trial budget was reached.


In [410]:
def required_ransac_trials(
    confidence,
    inlier_fraction,
    sample_size,
):
    success_probability = (
        inlier_fraction**sample_size
    )

    if success_probability >= 1.0:
        return 1

    if success_probability <= 0.0:
        return np.iinfo(
            np.int64
        ).max

    return math.ceil(
        np.log1p(-confidence)
        / np.log1p(
            -success_probability
        )
    )


def fit_ransac(
    Phi,
    y,
    sigma,
    threshold=3.0,
    confidence=0.999,
    max_trials=10_000,
    seed=0,
    return_info=False,
):
    Phi = np.asarray(
        Phi,
        dtype=float,
    )
    y = np.asarray(
        y,
        dtype=float,
    )

    n = len(y)
    sample_size = Phi.shape[1]

    if n < sample_size:
        raise ValueError(
            "RANSAC requires at least "
            f"{sample_size} observations"
        )

    rng = np.random.default_rng(
        seed
    )

    seen_subsets = set()
    total_subsets = math.comb(
        n,
        sample_size,
    )

    required_trials = min(
        total_subsets,
        max_trials,
    )

    best_mask = None
    best_count = -1
    best_error = np.inf
    trials = 0

    while (
        trials < required_trials
        and len(seen_subsets)
        < total_subsets
    ):
        subset = tuple(
            sorted(
                rng.choice(
                    n,
                    size=sample_size,
                    replace=False,
                )
            )
        )

        if subset in seen_subsets:
            continue

        seen_subsets.add(subset)
        trials += 1

        idx = np.asarray(subset)

        if (
            np.linalg.matrix_rank(
                Phi[idx]
            )
            < sample_size
        ):
            continue

        beta = weighted_least_squares(
            Phi[idx],
            y[idx],
        )

        residuals = np.abs(
            y - predict(beta, Phi)
        ) / sigma

        mask = (
            residuals
            <= threshold
        )

        count = int(
            mask.sum()
        )

        if count < sample_size:
            continue

        consensus_error = np.sum(
            residuals[mask] ** 2
        )

        better_model = (
            count > best_count
            or (
                count == best_count
                and consensus_error
                < best_error
            )
        )

        if better_model:
            best_mask = mask
            best_count = count
            best_error = (
                consensus_error
            )

            estimated_trials = (
                required_ransac_trials(
                    confidence,
                    count / n,
                    sample_size,
                )
            )

            required_trials = min(
                required_trials,
                max(
                    estimated_trials,
                    trials,
                ),
            )

    if best_mask is None:
        raise RuntimeError(
            "No valid RANSAC "
            "hypothesis was found"
        )

    beta_final = weighted_least_squares(
        Phi[best_mask],
        y[best_mask],
    )

    final_consensus_fraction = (
        best_count / n
    )

    theoretical_trials_from_final_consensus = (
        required_ransac_trials(
            confidence,
            final_consensus_fraction,
            sample_size,
        )
    )

    info = {
        "trials": trials,
        "consensus_fraction": (
            final_consensus_fraction
        ),
        "best_count": best_count,
        "sample_size": sample_size,
        "hit_max_trials": (
            trials >= max_trials
        ),
        "theoretical_trials_from_final_consensus": (
            theoretical_trials_from_final_consensus
        ),
    }

    if return_info:
        return beta_final, info

    return beta_final


### GNC-GM

Benchmark settings:

- $\bar{c}^{\,2}=\chi^2_{0.99,1}$
- division factor $1.4$
- maximum 100 continuation steps


In [411]:
def fit_gnc_gm(
    Phi,
    y,
    sigma,
    division_factor=1.4,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    c2 = chi2.ppf(
        0.99,
        df=1,
    )

    residuals = (
        normalized_squared_residuals(
            beta,
            Phi,
            y,
            sigma,
        )
    )

    mu = (
        2.0
        * residuals.max()
        / c2
    )

    if mu < 1.0:
        return beta

    for _ in range(max_iter):
        weights = (
            (c2 * mu)
            / (
                residuals
                + c2 * mu
            )
        ) ** 2

        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        mu /= division_factor

        if mu < 1.0:
            break

    return beta


### GNC-TLS

Benchmark settings:

- $\bar{c}^{\,2}=\chi^2_{0.99,1}$
- multiplication factor $1.4$
- tolerance $10^{-5}$
- maximum 100 iterations


In [412]:
def fit_gnc_tls(
    Phi,
    y,
    sigma,
    multiplication_factor=1.4,
    tol=1e-5,
    max_iter=100,
):
    beta = weighted_least_squares(
        Phi,
        y,
    )

    c2 = chi2.ppf(
        0.99,
        df=1,
    )

    residuals = (
        normalized_squared_residuals(
            beta,
            Phi,
            y,
            sigma,
        )
    )

    denominator = (
        2.0 * residuals.max()
        - c2
    )

    if denominator <= 1e-12:
        return beta

    mu = c2 / denominator

    previous_objective = (
        residuals.sum()
    )

    for _ in range(max_iter):
        upper = (
            (mu + 1.0)
            / mu
        ) * c2

        lower = (
            mu
            / (mu + 1.0)
        ) * c2

        weights = np.empty_like(
            residuals
        )

        high = residuals >= upper
        low = residuals <= lower
        middle = ~(high | low)

        weights[high] = 0.0
        weights[low] = 1.0

        weights[middle] = (
            np.sqrt(
                c2
                * mu
                * (mu + 1.0)
                / np.maximum(
                    residuals[middle],
                    1e-12,
                )
            )
            - mu
        )

        weights = np.clip(
            weights,
            0.0,
            1.0,
        )

        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        objective = np.sum(
            weights * residuals
        )

        relative_change = (
            abs(
                objective
                - previous_objective
            )
            / max(
                abs(
                    previous_objective
                ),
                1e-12,
            )
        )

        if (
            relative_change
            <= tol
        ):
            break

        previous_objective = (
            objective
        )

        mu *= (
            multiplication_factor
        )

    return beta


### ASOR

Benchmark settings:

- $a_0=0.5$
- $b_0=10^4$
- $A=10^4$
- $B=10^3$
- $\theta=0.5$
- tolerance $10^{-5}$
- maximum 100 iterations


In [413]:
def fit_asor(
    Phi,
    y,
    sigma,
    a0=0.5,
    b0=10_000.0,
    prior_a=10_000.0,
    prior_b=1_000.0,
    theta=0.5,
    tol=1e-5,
    max_iter=100,
):
    weights = np.ones(
        len(y)
    )

    b = b0
    alpha = a0 + 0.5

    zeta = (
        (1.0 / theta - 1.0)
        * gamma(alpha)
        / gamma(a0)
    )

    previous_objective = None

    beta = (
        weighted_least_squares(
            Phi,
            y,
        )
    )

    for _ in range(max_iter):
        beta = (
            weighted_least_squares(
                Phi,
                y,
                weights,
            )
        )

        residuals = (
            normalized_squared_residuals(
                beta,
                Phi,
                y,
                sigma,
            )
        )

        beta_i = (
            0.5 * residuals
            + b
        )

        log_term = (
            np.log(zeta)
            + a0
            * np.log(
                max(
                    b,
                    1e-12,
                )
            )
            - alpha
            * np.log(
                np.maximum(
                    beta_i,
                    1e-12,
                )
            )
            + 0.5
            * residuals
        )

        omega = expit(
            -log_term
        )

        numerator = (
            prior_a
            - 1.0
            + np.sum(
                a0
                * (
                    1.0
                    - omega
                )
            )
        )

        denominator = (
            prior_b
            + np.sum(
                (
                    1.0
                    - omega
                )
                * alpha
                / beta_i
            )
        )

        b = (
            numerator
            / denominator
        )

        new_weights = (
            omega
            + (
                1.0
                - omega
            )
            * alpha
            / beta_i
        )

        objective = np.sum(
            new_weights
            * residuals
        )

        if (
            previous_objective
            is not None
        ):
            relative_change = (
                abs(
                    objective
                    - previous_objective
                )
                / max(
                    abs(
                        previous_objective
                    ),
                    1e-12,
                )
            )

            if (
                relative_change
                <= tol
            ):
                weights = (
                    new_weights
                )
                break

        previous_objective = (
            objective
        )

        weights = new_weights

    return (
        weighted_least_squares(
            Phi,
            y,
            weights,
        )
    )


In [414]:
FITTERS = {
    "OLS": fit_ols,
    "Huber": fit_huber,
    "RANSAC": fit_ransac,
    "GNC-GM": fit_gnc_gm,
    "GNC-TLS": fit_gnc_tls,
    "ASOR": fit_asor,
}


## 5. Controlled Contamination

The Gaussian and uniform replacement mechanisms are parameterized as

$$
\mu_{\mathrm{out}}=\kappa_\mu\sigma_{\mathrm{in}},
\qquad
\sigma_{\mathrm{out}}^2=\kappa_\sigma\sigma_{\mathrm{in}}^2
$$

At corrupted indices, the nominal measurement error is **replaced**, not supplemented.

The coherent competing relationship is defined relative to the fixed domain reference $x=0$:

$$
f_{\mathrm{alt}}(x)=f(0)+5\left[f(x)-f(0)\right]
$$

For polynomial models, this leaves the constant term unchanged while amplifying the nonconstant structure by a factor of five. The same fixed construction is used for the spline so that the alternative relationship does not depend on the sampled predictor values.


In [415]:
def competing_curve(
    x,
    model_name,
    amplification=5.0,
):
    x = np.asarray(
        x,
        dtype=float,
    )

    nominal = true_curve(
        x,
        model_name,
    )

    reference_level = float(
        true_curve(
            np.array([0.0]),
            model_name,
        )[0]
    )

    return (
        reference_level
        + amplification
        * (
            nominal
            - reference_level
        )
    )


def generate_contaminated_data(
    model_name,
    n,
    outlier_ratio,
    variance_in,
    contamination_type,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    nominal_curve = true_curve(
        x,
        model_name,
    )

    sigma_in = np.sqrt(
        variance_in
    )

    y = (
        nominal_curve
        + rng.normal(
            0.0,
            sigma_in,
            size=n,
        )
    )

    n_outliers = round(
        outlier_ratio * n
    )

    outlier_mask = np.zeros(
        n,
        dtype=bool,
    )

    if n_outliers == 0:
        return (
            x,
            y,
            nominal_curve,
            outlier_mask,
        )

    selected = rng.choice(
        n,
        n_outliers,
        replace=False,
    )

    outlier_mask[selected] = True

    if (
        contamination_type
        == "zero_mean_gaussian"
    ):
        mean_out = 0.0
        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "biased_gaussian"
    ):
        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma_in
        )

        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "biased_uniform"
    ):
        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma_in
        )

        half_width = np.sqrt(
            3.0
            * VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.uniform(
            mean_out - half_width,
            mean_out + half_width,
            n_outliers,
        )

        y[selected] = (
            nominal_curve[selected]
            + outlier_noise
        )

    elif (
        contamination_type
        == "competing_curve"
    ):
        alternative = (
            competing_curve(
                x,
                model_name,
                amplification=5.0,
            )
        )

        y[selected] = (
            alternative[selected]
            + rng.normal(
                0.0,
                sigma_in,
                n_outliers,
            )
        )

    else:
        raise ValueError(
            "Unknown contamination type"
        )

    return (
        x,
        y,
        nominal_curve,
        outlier_mask,
    )


## 6. Prediction Error and One-Fit Wrapper

Prediction RMSE is evaluated against the noiseless nominal relationship on a dense grid:

$$
\mathrm{RMSE}_{\mathrm{pred}}
=
\sqrt{
\frac{1}{J}
\sum_{j=1}^{J}
\left[
\widehat{f}(x_j)-f(x_j)
\right]^2
}
$$

For the nominal-noise sweep, we also report

$$
\mathrm{NRMSE}
=
\frac{\mathrm{RMSE}_{\mathrm{pred}}}{\sigma_{\mathrm{in}}}
$$

Runtime is measured only for the estimator-fitting step after the design matrix has been constructed. Absolute timings are hardware dependent, so they should be interpreted primarily as relative comparisons obtained in the same execution environment.


In [416]:
def prediction_rmse(
    beta,
    model_name,
    x_eval,
):
    Phi_eval = design_matrix(
        x_eval,
        model_name,
    )

    y_true_eval = true_curve(
        x_eval,
        model_name,
    )

    y_hat_eval = predict(
        beta,
        Phi_eval,
    )

    return np.sqrt(
        np.mean(
            (
                y_hat_eval
                - y_true_eval
            ) ** 2
        )
    )


def fit_method(
    method,
    Phi,
    y,
    sigma,
    seed,
):
    diagnostics = {
        "ransac_trials": np.nan,
        "ransac_consensus_fraction": np.nan,
        "ransac_hit_max_trials": False,
        "ransac_theoretical_trials_final": np.nan,
    }

    start = time.perf_counter()

    if method == "OLS":
        beta = fit_ols(
            Phi,
            y,
        )

    elif method == "Huber":
        beta = fit_huber(
            Phi,
            y,
            sigma,
        )

    elif method == "RANSAC":
        beta, info = fit_ransac(
            Phi,
            y,
            sigma,
            threshold=3.0,
            confidence=0.999,
            max_trials=10_000,
            seed=seed,
            return_info=True,
        )

        diagnostics.update({
            "ransac_trials": (
                info["trials"]
            ),
            "ransac_consensus_fraction": (
                info[
                    "consensus_fraction"
                ]
            ),
            "ransac_hit_max_trials": (
                info[
                    "hit_max_trials"
                ]
            ),
            "ransac_theoretical_trials_final": (
                info[
                    "theoretical_trials_from_final_consensus"
                ]
            ),
        })

    elif method == "GNC-GM":
        beta = fit_gnc_gm(
            Phi,
            y,
            sigma,
        )

    elif method == "GNC-TLS":
        beta = fit_gnc_tls(
            Phi,
            y,
            sigma,
        )

    elif method == "ASOR":
        beta = fit_asor(
            Phi,
            y,
            sigma,
        )

    else:
        raise ValueError(
            f"Unknown method: {method}"
        )

    runtime = (
        time.perf_counter()
        - start
    )

    return (
        beta,
        runtime,
        diagnostics,
    )


## 7. Sanity Check

This is deliberately small. Run it before the complete benchmark.


In [417]:
x_eval = np.linspace(
    X_MIN,
    X_MAX,
    1000,
)

sanity_rows = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    seed = (
        MASTER_SEED
        + 10000 * model_index
    )

    x, y, _, _ = (
        generate_contaminated_data(
            model_name=model_name,
            n=100,
            outlier_ratio=0.30,
            variance_in=1000.0,
            contamination_type=(
                "biased_gaussian"
            ),
            seed=seed,
        )
    )

    Phi = design_matrix(
        x,
        model_name,
    )

    sigma = np.sqrt(1000.0)

    for method_index, method in enumerate(
        METHOD_NAMES
    ):
        beta, runtime, diagnostics = fit_method(
            method,
            Phi,
            y,
            sigma,
            seed=(
                seed
                + method_index
            ),
        )

        rmse = prediction_rmse(
            beta,
            model_name,
            x_eval,
        )

        sanity_rows.append({
            "model": model_name,
            "method": method,
            "RMSE_pred": rmse,
            "runtime_s": runtime,
            **diagnostics,
        })

sanity = pd.DataFrame(
    sanity_rows
)

sanity[
    [
        "model",
        "method",
        "RMSE_pred",
        "runtime_s",
    ]
]


,model,method,RMSE_pred,runtime_s
0,quadratic,OLS,30.126818,0.000137
1,quadratic,Huber,17.859206,0.000865
2,quadratic,RANSAC,4.872127,0.002581
3,quadratic,GNC-GM,15.197143,0.000779
4,quadratic,GNC-TLS,13.068138,0.001502
5,quadratic,ASOR,10.675353,0.003136
6,cubic,OLS,34.648527,0.000075
7,cubic,Huber,23.435721,0.000815
8,cubic,RANSAC,12.487155,0.002380
9,cubic,GNC-GM,19.672791,0.000704


The sanity-check table shows only metrics that apply to every estimator. RANSAC-specific diagnostics are reported separately in the dedicated RANSAC section.

## 8. Monte Carlo Engine

Within each Monte Carlo realization, **all six estimators receive exactly the same generated dataset**.


In [418]:
def run_condition(
    model_name,
    n,
    outlier_ratio,
    variance_in,
    contamination_type,
    mc_runs,
    seed_offset=0,
):
    sigma = np.sqrt(
        variance_in
    )

    x_eval = np.linspace(
        X_MIN,
        X_MAX,
        1200,
    )

    rows = []

    for mc in range(mc_runs):
        seed = (
            MASTER_SEED
            + seed_offset
            + 100003 * mc
            + 1009 * n
            + int(
                1000
                * outlier_ratio
            )
        )

        x, y, _, outlier_mask = (
            generate_contaminated_data(
                model_name=model_name,
                n=n,
                outlier_ratio=(
                    outlier_ratio
                ),
                variance_in=(
                    variance_in
                ),
                contamination_type=(
                    contamination_type
                ),
                seed=seed,
            )
        )

        Phi = design_matrix(
            x,
            model_name,
        )

        for method_index, method in enumerate(
            METHOD_NAMES
        ):
            beta, runtime, diagnostics = fit_method(
                method,
                Phi,
                y,
                sigma,
                seed=(
                    seed
                    + method_index
                ),
            )

            rmse = prediction_rmse(
                beta,
                model_name,
                x_eval,
            )

            rows.append({
                "model": model_name,
                "method": method,
                "contamination_type": (
                    contamination_type
                ),
                "outlier_ratio": (
                    outlier_ratio
                ),
                "N": n,
                "variance_in": (
                    variance_in
                ),
                "sigma_in": sigma,
                "mc": mc,
                "RMSE_pred": rmse,
                "NRMSE": (
                    rmse / sigma
                ),
                "runtime_s": runtime,
                "actual_outliers": int(
                    outlier_mask.sum()
                ),
                **diagnostics,
            })

    return pd.DataFrame(
        rows
    )


def summarize_metric(
    data,
    group_columns,
    metric,
):
    return (
        data
        .groupby(
            group_columns
        )[metric]
        .agg(
            median="median",
            mean="mean",
            std="std",
            q25=lambda s: (
                s.quantile(0.25)
            ),
            q75=lambda s: (
                s.quantile(0.75)
            ),
            count="count",
        )
        .reset_index()
    )


## 9. Contamination Sweep

Main configuration:

- $N=100$
- $\sigma_{\mathrm{in}}^2=1000$

The four contamination models are:

1. zero-mean Gaussian replacement with $\kappa_\sigma=10$
2. biased Gaussian replacement with $\kappa_\sigma=10$ and $\kappa_\mu=3$
3. biased uniform replacement with the same mean and variance as case 2
4. a coherent competing curve


In [419]:
contamination_frames = []

for contamination_index, contamination_type in enumerate(
    CONTAMINATION_TYPES
):
    for model_index, model_name in enumerate(
        MODEL_NAMES
    ):
        for ratio in CONTAMINATION_RATIOS:
            result = run_condition(
                model_name=model_name,
                n=BASELINE_N,
                outlier_ratio=float(ratio),
                variance_in=BASELINE_VARIANCE,
                contamination_type=contamination_type,
                mc_runs=MC_RUNS,
                seed_offset=(
                    10_000_000 * contamination_index
                    + 1_000_000 * model_index
                ),
            )
            contamination_frames.append(result)

contamination_results = pd.concat(
    contamination_frames,
    ignore_index=True,
)

contamination_summary = summarize_metric(
    contamination_results,
    [
        "model",
        "contamination_type",
        "outlier_ratio",
        "method",
    ],
    "NRMSE",
)

contamination_runtime_summary = summarize_metric(
    contamination_results,
    [
        "model",
        "contamination_type",
        "outlier_ratio",
        "method",
    ],
    "runtime_s",
)

print(
    f"Contamination sweep complete: "
    f"{len(contamination_results):,} estimator fits."
)


Contamination sweep complete: 12,960 estimator fits.


In [420]:
# Article table: B-spline under 50% zero-mean Gaussian replacement
# This supports the statement that estimator ranking depends on contamination structure.

zero_mean_bspline_table = (
    contamination_results
    .query(
        "model == 'bspline' "
        "and contamination_type == 'zero_mean_gaussian' "
        "and outlier_ratio == 0.50"
    )
    .groupby("method")["NRMSE"]
    .median()
    .reindex(METHOD_NAMES)
    .rename("Median NRMSE")
    .to_frame()
)

zero_mean_bspline_table


,Median NRMSE
method,
OLS,0.691457
Huber,0.525695
RANSAC,2.350238
GNC-GM,0.540148
GNC-TLS,0.579410
ASOR,0.554022


## 10. Sample-Size Scaling

Configuration:

- $50\%$ biased Gaussian contamination
- $\sigma_{\mathrm{in}}^2=1000$
- $\kappa_\sigma=10$
- $\kappa_\mu=3$
- $N\in\{50,100,200,500,1000,2000,5000\}$


In [421]:
sample_frames = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    for n in SAMPLE_SIZES:
        result = run_condition(
            model_name=model_name,
            n=n,
            outlier_ratio=0.50,
            variance_in=1000.0,
            contamination_type="biased_gaussian",
            mc_runs=MC_RUNS,
            seed_offset=(
                30_000_000
                + 1_000_000 * model_index
            ),
        )
        sample_frames.append(result)

sample_results = pd.concat(
    sample_frames,
    ignore_index=True,
)

sample_summary = summarize_metric(
    sample_results,
    ["model", "N", "method"],
    "NRMSE",
)

sample_runtime_summary = summarize_metric(
    sample_results,
    ["model", "N", "method"],
    "runtime_s",
)

print(
    f"Sample-size sweep complete: "
    f"{len(sample_results):,} estimator fits."
)


Sample-size sweep complete: 3,780 estimator fits.


In [422]:
# B-spline sample-size scaling under 50% biased Gaussian contamination
# Show every estimator and every sample size executed by the selected profile.

sample_size_article_table = (
    sample_results[
        sample_results["model"] == "bspline"
    ]
    .groupby(["method", "N"])["NRMSE"]
    .median()
    .unstack("N")
    .reindex(METHOD_NAMES)
    .reindex(columns=SAMPLE_SIZES)
)

sample_size_article_table.columns.name = "N"
sample_size_article_table.index.name = "Method"

if RUN_PROFILE != "full":
    print(
        "Quick profile: only N = 50, 100, and 500 are evaluated. "
        "Set RUN_PROFILE='full' to display all seven sample sizes: "
        "50, 100, 200, 500, 1000, 2000, and 5000."
    )

sample_size_article_table


N,50,100,200,500,1000,2000,5000
Method,,,,,,,
OLS,2.068496,1.635838,1.590575,1.538392,1.534328,1.504890,1.504037
Huber,1.755077,1.194282,1.090520,0.892807,0.905359,0.855868,0.863769
RANSAC,3.915899,2.775842,1.395150,1.656156,1.316123,1.462962,1.437782
GNC-GM,1.703386,0.944055,0.778396,0.611273,0.606441,0.564576,0.544942
GNC-TLS,1.641214,0.785673,0.628569,0.360374,0.328141,0.280967,0.275299
ASOR,1.707309,0.776272,0.674134,0.460562,0.445280,0.401429,0.406641


## 11. Nominal-Noise Scaling

Configuration:

- $N=500$
- $50\%$ biased Gaussian contamination
- $\kappa_\sigma=10$
- $\kappa_\mu=3$
- $\sigma_{\mathrm{in}}^2\in\{10,100,1000,10000\}$

Each noise variance receives a different Monte Carlo seed stream. This prevents the NRMSE values from becoming numerically identical simply because the same standardized random realization has been rescaled.


In [423]:
noise_frames = []

for model_index, model_name in enumerate(
    MODEL_NAMES
):
    for noise_index, variance_in in enumerate(
        NOISE_VARIANCES
    ):
        result = run_condition(
            model_name=model_name,
            n=500,
            outlier_ratio=0.50,
            variance_in=float(variance_in),
            contamination_type="biased_gaussian",
            mc_runs=MC_RUNS,
            seed_offset=(
                50_000_000
                + 1_000_000 * model_index
                + 100_000 * noise_index
            ),
        )
        noise_frames.append(result)

noise_results = pd.concat(
    noise_frames,
    ignore_index=True,
)

noise_summary = summarize_metric(
    noise_results,
    [
        "model",
        "variance_in",
        "method",
    ],
    "NRMSE",
)

print(
    f"Noise sweep complete: "
    f"{len(noise_results):,} estimator fits."
)


Noise sweep complete: 2,160 estimator fits.


In [424]:
# Article table: normalized error across nominal-noise scales
# The article discusses the broad stability of NRMSE when all scales are changed consistently.

noise_article_table = (
    noise_results[
        noise_results["method"].isin(
            ["GNC-TLS", "ASOR"]
        )
    ]
    .groupby(
        ["model", "method", "variance_in"]
    )["NRMSE"]
    .median()
    .unstack("variance_in")
)

noise_article_table


variance_in         10.0      100.0     1000.0    10000.0
model     method                                         
bspline   ASOR     0.458744  0.523278  0.469927  0.481698
          GNC-TLS  0.363098  0.410741  0.380853  0.389872
cubic     ASOR     0.449008  0.447932  0.403896  0.413067
          GNC-TLS  0.324752  0.303328  0.274466  0.295001
quadratic ASOR     0.419994  0.439057  0.438193  0.405305
          GNC-TLS  0.275029  0.311246  0.319179  0.287108

## 12. RANSAC Minimal-Sample Burden

For the standard idealized RANSAC calculation, let $s$ be the minimal sample size, $w$ the inlier fraction, and $p$ the desired confidence. The required number of hypotheses is

$$
K
=
\left\lceil
\frac{\log(1-p)}
{\log\left(1-w^s\right)}
\right\rceil
$$

This provides a theoretical reference for how the all-inlier sampling burden grows with model dimension. The implementation samples distinct subsets and adapts its stopping budget from the best consensus found so far, so its realized trial count need not equal this idealized value.

The empirical contamination results also record:

- `ransac_trials`
- `ransac_consensus_fraction`
- `ransac_hit_max_trials`
- `ransac_theoretical_trials_final`

These diagnostics help distinguish poor statistical performance from cases in which the imposed 10,000-hypothesis budget is exhausted.


In [425]:
ransac_rows = []

for model_name in MODEL_NAMES:
    sample_size = number_of_coefficients(
        model_name
    )

    for ratio in CONTAMINATION_RATIOS_FULL:
        inlier_fraction = 1.0 - ratio

        trials = required_ransac_trials(
            confidence=0.999,
            inlier_fraction=inlier_fraction,
            sample_size=sample_size,
        )

        ransac_rows.append({
            "model": model_name,
            "sample_size": sample_size,
            "outlier_ratio": ratio,
            "required_trials": trials,
        })

ransac_theory = pd.DataFrame(
    ransac_rows
)

ransac_theory.head()


,model,sample_size,outlier_ratio,required_trials
0,quadratic,3,0.0,1
1,quadratic,3,0.1,6
2,quadratic,3,0.3,17
3,quadratic,3,0.5,52
4,quadratic,3,0.7,253


In [426]:
# Article table: RANSAC model dimension, theoretical burden,
# measured runtime, and measured prediction error at 50% biased Gaussian contamination.

ransac_empirical_50 = (
    contamination_results[
        (contamination_results["method"] == "RANSAC")
        & (
            contamination_results["contamination_type"]
            == "biased_gaussian"
        )
        & (contamination_results["outlier_ratio"] == 0.50)
    ]
    .groupby("model")
    .agg(
        median_runtime_s=("runtime_s", "median"),
        median_NRMSE=("NRMSE", "median"),
        median_trials=("ransac_trials", "median"),
    )
)

ransac_theory_50 = (
    ransac_theory[
        ransac_theory["outlier_ratio"] == 0.50
    ]
    .set_index("model")[
        ["sample_size", "required_trials"]
    ]
)

ransac_article_table = (
    ransac_theory_50
    .join(ransac_empirical_50)
    .rename(columns={
        "sample_size": "Minimal sample s",
        "required_trials": "Theoretical trials",
        "median_runtime_s": "Median runtime (s)",
        "median_NRMSE": "Median NRMSE",
        "median_trials": "Median actual trials",
    })
    .reindex(MODEL_NAMES)
)

ransac_article_table


,Minimal sample s,Theoretical trials,Median runtime (s),Median NRMSE,Median actual trials
model,,,,,
quadratic,3,52,0.002055,0.641979,15.0
cubic,4,108,0.003295,0.811817,25.0
bspline,8,1765,0.016983,1.838730,184.5


### RANSAC Interpretation

The theoretical trial count is the standard idealized all-inlier minimal-sample calculation. The measured trial count can differ because the implementation uses distinct subsets and adaptive stopping.


In [427]:
# Optional diagnostic: fraction of RANSAC realizations that reached the 10,000-trial cap

ransac_cap_table = (
    contamination_results[
        contamination_results["method"] == "RANSAC"
    ]
    .groupby(
        [
            "model",
            "contamination_type",
            "outlier_ratio",
        ]
    )["ransac_hit_max_trials"]
    .mean()
    .rename("Fraction hitting trial cap")
    .to_frame()
)

ransac_cap_table[
    ransac_cap_table[
        "Fraction hitting trial cap"
    ] > 0
]


,,,Fraction hitting trial cap
model,contamination_type,outlier_ratio,


## 13. Common-Truth Model-Richness Experiment

The main benchmark uses a different nominal curve for each model class. That is useful for studying robustness within each representation, but it does not completely isolate what happens when the fitted representation changes.

This controlled experiment therefore generates **one quadratic ground truth** and fits all three bases to the same generated dataset in each Monte Carlo realization:

- quadratic polynomial — 3 coefficients
- cubic polynomial — 4 coefficients
- quadratic B-spline — 8 coefficients

The predictor values, observations, outlier locations, and noise realization are identical across the three fits.

A cubic polynomial basis and the quadratic B-spline basis can both represent a quadratic function over the benchmark domain. The comparison is therefore not confounded by a different ground-truth curve; remaining differences reflect the fitted representation—including its dimension and basis geometry—together with estimator behavior.

The contamination model is biased Gaussian replacement with $\kappa_\sigma=10$ and $\kappa_\mu=3$.


In [428]:
COMMON_TRUTH_BETA = np.array([
    20.0,
    260.0,
    120.0,
])


def shared_quadratic_truth(x):
    return (
        polynomial_design(
            x,
            degree=2,
        )
        @ COMMON_TRUTH_BETA
    )


def generate_common_truth_data(
    n,
    outlier_ratio,
    variance_in,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    y_true = shared_quadratic_truth(
        x
    )

    sigma = np.sqrt(
        variance_in
    )

    y = (
        y_true
        + rng.normal(
            0.0,
            sigma,
            n,
        )
    )

    n_outliers = round(
        outlier_ratio * n
    )

    outlier_mask = np.zeros(
        n,
        dtype=bool,
    )

    if n_outliers > 0:
        selected = rng.choice(
            n,
            n_outliers,
            replace=False,
        )

        outlier_mask[selected] = True

        mean_out = (
            BIASED_MEAN_SHIFT_SIGMA
            * sigma
        )

        variance_out = (
            VARIANCE_RATIO
            * variance_in
        )

        outlier_noise = rng.normal(
            mean_out,
            np.sqrt(
                variance_out
            ),
            n_outliers,
        )

        y[selected] = (
            y_true[selected]
            + outlier_noise
        )

    return (
        x,
        y,
        y_true,
        outlier_mask,
    )


def prediction_rmse_against_shared_truth(
    beta,
    model_name,
    x_eval,
):
    Phi_eval = design_matrix(
        x_eval,
        model_name,
    )

    y_hat = Phi_eval @ beta
    y_true = shared_quadratic_truth(
        x_eval
    )

    return np.sqrt(
        np.mean(
            (
                y_hat
                - y_true
            ) ** 2
        )
    )


In [429]:
common_truth_ratios = (
    CONTAMINATION_RATIOS_FULL
    if RUN_PROFILE == "full"
    else np.array([
        0.0,
        0.30,
        0.50,
    ])
)

common_truth_rows = []
x_eval_common = np.linspace(
    X_MIN,
    X_MAX,
    1200,
)

for ratio in common_truth_ratios:
    for mc in range(MC_RUNS):
        seed = (
            MASTER_SEED
            + 70_000_000
            + 100003 * mc
            + int(1000 * ratio)
        )

        x, y, _, outlier_mask = (
            generate_common_truth_data(
                n=BASELINE_N,
                outlier_ratio=float(ratio),
                variance_in=BASELINE_VARIANCE,
                seed=seed,
            )
        )

        sigma = np.sqrt(
            BASELINE_VARIANCE
        )

        for model_index, model_name in enumerate(
            MODEL_NAMES
        ):
            Phi = design_matrix(
                x,
                model_name,
            )

            for method_index, method in enumerate(
                METHOD_NAMES
            ):
                beta, runtime, diagnostics = fit_method(
                    method,
                    Phi,
                    y,
                    sigma,
                    seed=(
                        seed
                        + 1000 * model_index
                        + method_index
                    ),
                )

                rmse = (
                    prediction_rmse_against_shared_truth(
                        beta,
                        model_name,
                        x_eval_common,
                    )
                )

                common_truth_rows.append({
                    "model": model_name,
                    "method": method,
                    "outlier_ratio": float(ratio),
                    "N": BASELINE_N,
                    "variance_in": BASELINE_VARIANCE,
                    "mc": mc,
                    "RMSE_pred": rmse,
                    "NRMSE": rmse / sigma,
                    "runtime_s": runtime,
                    "actual_outliers": int(
                        outlier_mask.sum()
                    ),
                    **diagnostics,
                })

common_truth_results = pd.DataFrame(
    common_truth_rows
)

common_truth_summary = summarize_metric(
    common_truth_results,
    [
        "model",
        "outlier_ratio",
        "method",
    ],
    "NRMSE",
)

print(
    f"Common-truth experiment complete: "
    f"{len(common_truth_results):,} estimator fits."
)


Common-truth experiment complete: 3,240 estimator fits.


In [430]:
# Article table: same quadratic ground truth, same N=100 data,
# 30% biased Gaussian contamination, different fitted representations.

model_richness_table = (
    common_truth_results[
        common_truth_results["outlier_ratio"] == 0.30
    ]
    .groupby(["method", "model"])["NRMSE"]
    .median()
    .unstack("model")
    .reindex(METHOD_NAMES)
    .reindex(columns=MODEL_NAMES)
)

model_richness_table.index.name = "Method"
model_richness_table.columns = [
    "Quadratic",
    "Cubic",
    "B-spline",
]

model_richness_table


,Quadratic,Cubic,B-spline
Method,,,
OLS,0.934498,0.936756,1.131738
Huber,0.451037,0.496860,0.673959
RANSAC,0.273277,0.382717,1.384040
GNC-GM,0.293039,0.347691,0.492866
GNC-TLS,0.245916,0.276208,0.484893
ASOR,0.257085,0.307264,0.469004


## 14. Illustrative Local-Support Experiment

This single-realization experiment asks a narrower question: when contamination is localized in $x$, how does a locally supported spline basis change the way deformation spreads across the fitted curve?

The nominal truth is quadratic so that both a cubic polynomial basis and the quadratic B-spline basis can represent it.

Only $50\%$ of the measurements inside $[-20,20]$ are replaced. The remaining measurements in the same interval stay nominal, so clean observations still support the local relationship.

Both OLS and ASOR are shown to separate the effect of the basis from the effect of robust weighting. Because this section uses one illustrative realization rather than the 30-run Monte Carlo benchmark, its numerical values should not be treated as distributional performance claims.


In [431]:
def generate_localized_contamination(
    n=500,
    variance_in=1000.0,
    x_band=(-20.0, 20.0),
    local_contamination_fraction=0.50,
    mean_shift_sigma=3.0,
    variance_ratio=10.0,
    seed=MASTER_SEED + 777,
):
    rng = np.random.default_rng(
        seed
    )

    x = rng.uniform(
        X_MIN,
        X_MAX,
        n,
    )

    y_true = shared_quadratic_truth(
        x
    )

    sigma = np.sqrt(
        variance_in
    )

    y = (
        y_true
        + rng.normal(
            0.0,
            sigma,
            n,
        )
    )

    local_candidates = np.flatnonzero(
        (x >= x_band[0])
        & (x <= x_band[1])
    )

    n_local_outliers = round(
        local_contamination_fraction
        * len(local_candidates)
    )

    contaminated = np.zeros(
        n,
        dtype=bool,
    )

    if n_local_outliers > 0:
        selected = rng.choice(
            local_candidates,
            n_local_outliers,
            replace=False,
        )

        contaminated[selected] = True

        outlier_noise = rng.normal(
            mean_shift_sigma * sigma,
            np.sqrt(
                variance_ratio
                * variance_in
            ),
            n_local_outliers,
        )

        y[selected] = (
            y_true[selected]
            + outlier_noise
        )

    return (
        x,
        y,
        y_true,
        contaminated,
        local_candidates,
    )


x_local, y_local, _, local_mask, local_candidates = (
    generate_localized_contamination()
)

x_eval_local = np.linspace(
    X_MIN,
    X_MAX,
    1400,
)

truth_eval_local = (
    shared_quadratic_truth(
        x_eval_local
    )
)

local_rows = []
local_predictions = {}

for model_name in [
    "cubic",
    "bspline",
]:
    Phi = design_matrix(
        x_local,
        model_name,
    )

    Phi_eval = design_matrix(
        x_eval_local,
        model_name,
    )

    for method in [
        "OLS",
        "ASOR",
    ]:
        beta, runtime, diagnostics = fit_method(
            method,
            Phi,
            y_local,
            np.sqrt(1000.0),
            seed=MASTER_SEED + 888,
        )

        prediction = (
            Phi_eval @ beta
        )

        local_predictions[
            (model_name, method)
        ] = prediction

        near = (
            (x_eval_local >= -20.0)
            & (x_eval_local <= 20.0)
        )

        far = ~near

        local_rows.append({
            "model": model_name,
            "method": method,
            "RMSE_all": np.sqrt(
                np.mean(
                    (
                        prediction
                        - truth_eval_local
                    ) ** 2
                )
            ),
            "RMSE_local_region": np.sqrt(
                np.mean(
                    (
                        prediction[near]
                        - truth_eval_local[near]
                    ) ** 2
                )
            ),
            "RMSE_away_from_region": np.sqrt(
                np.mean(
                    (
                        prediction[far]
                        - truth_eval_local[far]
                    ) ** 2
                )
            ),
            "runtime_s": runtime,
            "local_candidate_count": int(
                len(local_candidates)
            ),
            "local_outlier_count": int(
                local_mask.sum()
            ),
            "local_contamination_fraction_realized": (
                local_mask.sum()
                / max(
                    len(local_candidates),
                    1,
                )
            ),
        })

local_support_results = pd.DataFrame(
    local_rows
)

local_support_results[
    [
        "model",
        "method",
        "RMSE_local_region",
        "RMSE_away_from_region",
    ]
]


,model,method,RMSE_local_region,RMSE_away_from_region
0,cubic,OLS,22.575510,13.585273
1,cubic,ASOR,7.019934,4.794329
2,bspline,OLS,40.893478,12.079083
3,bspline,ASOR,12.554665,5.507550


In [432]:
# Article table: illustrative local-support comparison

local_support_article_table = (
    local_support_results[
        [
            "model",
            "method",
            "RMSE_local_region",
            "RMSE_away_from_region",
        ]
    ]
    .rename(columns={
        "model": "Model",
        "method": "Estimator",
        "RMSE_local_region": "RMSE inside contaminated region",
        "RMSE_away_from_region": "RMSE outside contaminated region",
    })
)

local_support_article_table


,Model,Estimator,RMSE inside contaminated region,RMSE outside contaminated region
0,cubic,OLS,22.575510,13.585273
1,cubic,ASOR,7.019934,4.794329
2,bspline,OLS,40.893478,12.079083
3,bspline,ASOR,12.554665,5.507550


## 15. Competing-Curve Check

The article treats the coherent competing relationship as an identifiability problem near equal support. The compact table below shows the spread of prediction error for the quadratic model around that transition.


In [433]:
competing_curve_article_table = (
    contamination_results[
        (contamination_results["model"] == "quadratic")
        & (
            contamination_results["contamination_type"]
            == "competing_curve"
        )
        & (
            contamination_results["outlier_ratio"]
            .isin([0.30, 0.50, 0.70])
        )
        & (
            contamination_results["method"]
            .isin(["RANSAC", "GNC-GM", "GNC-TLS", "ASOR"])
        )
    ]
    .groupby(
        ["outlier_ratio", "method"]
    )["NRMSE"]
    .agg(
        q25=lambda s: s.quantile(0.25),
        median="median",
        q75=lambda s: s.quantile(0.75),
    )
    .round(3)
)

if RUN_PROFILE != "full":
    print(
        "Quick profile does not include 70% contamination. "
        "Set RUN_PROFILE='full' for the complete article table."
    )

competing_curve_article_table


q25  median     q75
outlier_ratio method                         
0.3           ASOR      0.140   0.200   0.271
              GNC-GM    0.145   0.192   0.277
              GNC-TLS   0.124   0.189   0.262
              RANSAC    0.145   0.214   0.291
0.5           ASOR      8.091   9.758  18.320
              GNC-GM    7.397   9.162  18.344
              GNC-TLS   7.727   9.770  18.458
              RANSAC    0.303   0.481  20.185
0.7           ASOR     19.998  20.119  20.242
              GNC-GM   19.989  20.141  20.284
              GNC-TLS  20.060  20.174  20.297
              RANSAC   20.084  20.157  20.271

## 16. Article-Supporting Tables

The notebook intentionally displays the numerical summaries used to support the article rather than producing a large collection of plots or CSV files.


In [434]:
article_tables = {
    "model_richness_30pct_biased_gaussian": model_richness_table,
    "ransac_50pct_biased_gaussian": ransac_article_table,
    "bspline_sample_size_50pct_biased_gaussian": sample_size_article_table,
    "bspline_50pct_zero_mean_gaussian": zero_mean_bspline_table,
    "noise_scaling": noise_article_table,
    "competing_curve_transition": competing_curve_article_table,
    "local_support_illustration": local_support_article_table,
}

print("Article-supporting tables available:")
for name in article_tables:
    print(f"  - {name}")


Article-supporting tables available:
  - model_richness_30pct_biased_gaussian
  - ransac_50pct_biased_gaussian
  - bspline_sample_size_50pct_biased_gaussian
  - bspline_50pct_zero_mean_gaussian
  - noise_scaling
  - competing_curve_transition
  - local_support_illustration


## 17. Key Findings from the Full Benchmark

The article's quantitative conclusions are based on the **full profile**, with 30 Monte Carlo realizations per exact condition.

### Model richness at fixed data size

In the controlled common-truth experiment, all three representations fit the same $N=100$ observations in each realization. At $30\%$ biased Gaussian contamination, GNC-TLS and ASOR degrade only modestly from the quadratic to the cubic representation, while the eight-function B-spline is more demanding at the same sample size. RANSAC is affected more strongly.

This should **not** be interpreted as evidence that splines are inherently less robust. The fitted representations differ in dimension and basis geometry.

### RANSAC and model dimension

RANSAC's minimal sample size increases from 3 to 4 to 8 observations for the quadratic, cubic, and B-spline fits. The idealized all-inlier trial burden therefore grows rapidly with model dimension, and the measured runtime rises substantially for the B-spline in this benchmark.

### More data helps the richer representation

Under $50\%$ biased Gaussian contamination, the B-spline becomes substantially more stable as $N$ increases. Increasing sample size reduces estimation variability for the robust methods, although it does not remove the systematic effect of biased contamination on OLS.

### The contamination pattern matters

There is no universal estimator ranking. Under biased contamination, GNC-TLS and ASOR generally perform strongly through moderate contamination, with GNC-GM often close behind. Under zero-mean Gaussian replacement, the differences among the soft robust estimators are smaller.

### Local support changes where deformation appears

The local-support experiment is a **single illustrative realization**, not a 30-run Monte Carlo result. It supports the limited conclusion that local support can change how deformation is distributed spatially; it does not show that a spline is automatically more robust.

### A coherent competing curve becomes an identifiability problem

Near equal support for two coherent relationships, different Monte Carlo realizations may favor different structures. Once the competing relationship becomes the majority structure, robust single-model estimators generally follow it. This is an identifiability limitation, not merely an optimization failure.

### Overall interpretation

> **The basis determines what shapes the model can represent. The robust estimator determines how much each observation is trusted.**

A richer basis changes representation, parameter burden, and locality. Robust estimation changes sensitivity to contaminated observations. Neither mechanism substitutes for the other.


## 18. Reproducibility Notes

For the numerical values reported in the article, set

```python
RUN_PROFILE = "full"
```

The full profile uses 30 Monte Carlo realizations per exact condition.

The `"quick"` profile is only a smoke test. Its tables should **not** be compared numerically with the article.

Runtime is hardware dependent and should be interpreted comparatively within the same computing environment.

The publication-quality figures used in the article are kept separately in the repository's `figures/` folder; this notebook focuses on the algorithms, experiments, and numerical tables needed to support the reported conclusions.
